# Nemotron LoRA — train on Kaggle (free 2×T4, 4-bit QLoRA)

**Reality check:** the T4 (sm_75) can't run the fused Mamba kernel, so it uses the
pure-PyTorch `torch_forward` path — correct, but **slow (~20–60 s/step)**. One 12 h
session trains only a *fraction* of an epoch. The adapter is saved to
`/kaggle/working/lora_adapter` every few steps, and you can **resume across sessions**
(see the last section) to keep improving it for free.


## 1. Code + dependencies (keep Kaggle's torch 2.10)

In [ ]:
%cd /kaggle/working
!rm -rf repo && git clone -b build/nemotron-pipeline https://github.com/SebAustin/NVIDIA-Nemotron-Model-Reasoning-Challenge repo
%cd repo
!pip install -q "transformers>=4.45,<5" peft trl datasets accelerate bitsandbytes psutil einops
# we never use vision; a mismatched torchvision breaks transformers' import -> remove it
!pip uninstall -y -q torchvision torchaudio
import torch
print("TORCH:", torch.__version__, "abi:", torch.compiled_with_cxx11_abi())

## 2. Install mamba_ssm only (torch_forward path; skips the T4-incompatible SSD kernel)

In [ ]:
# Install ONLY mamba_ssm (needed at import for rmsnorm), NOT causal_conv1d.
# Without causal_conv1d the model uses its torch_forward path, which skips the
# Mamba-2 SSD Triton kernel that fails to compile on T4 (sm_75).
!pip install -q --no-deps 'https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl'
!python -c "import mamba_ssm; print('mamba_ssm OK; causal_conv1d intentionally absent -> torch_forward')"

## 3. Competition data (recursive find)

In [ ]:
import os, glob, shutil, urllib.request
os.makedirs('data', exist_ok=True)
hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
if hits:
    shutil.copy(hits[0], 'data/train.csv'); print('train.csv (mount) <-', hits[0])
else:
    TOK = "KGAT_xxxxxxxxxxxxxxxx"   # <-- your Kaggle API token (Colab/local path)
    url='https://www.kaggle.com/api/v1/competitions/data/download/nvidia-nemotron-model-reasoning-challenge/train.csv'
    req=urllib.request.Request(url, headers={'Authorization': f'Bearer {TOK}'})
    open('data/train.csv','wb').write(urllib.request.urlopen(req).read())
    print('train.csv (download):', os.path.getsize('data/train.csv'), 'bytes')

## 4. EDA + build the SFT data

In [ ]:
!python scripts/01_eda.py --data-dir data
!python scripts/02_prepare_data.py --data-dir data

## 5. Train (4-bit QLoRA, model split across both T4s)

- `device_map='auto'` + `max_memory` spreads the 17 GB 4-bit model across the two
  16 GB T4s (+ CPU offload if needed).
- `torch_forward` is auto-forced on sm_75; `GRAD_CHECKPOINT=1` keeps memory in budget.
- Small `GRAD_ACCUM` so optimizer steps (and checkpoints) come quickly on a slow GPU.
- `RESUME_ADAPTER`: if you attach a previous run's adapter as a Kaggle **dataset**,
  set it below to continue training those weights instead of starting fresh.


In [ ]:
import os, glob
os.environ['QUANT'] = '4bit'
os.environ['NEMOTRON_MAX_MEMORY_GPU'] = '13GiB'   # per T4 (16 GB) — leave headroom
os.environ['SFT_MAX_SEQ_LENGTH'] = '512'           # data is short; faster + less memory
os.environ['GRAD_ACCUM'] = '4'                     # quicker optimizer steps on a slow GPU
os.environ['SAVE_STEPS'] = '25'                    # bank the adapter ~every 100 micro-steps
os.environ['GRAD_CHECKPOINT'] = '1'                # needed to fit 16 GB/card
os.environ['NUM_EPOCHS'] = '1'

# --- resume across sessions: point at a previously-saved adapter if you attached one ---
prev = glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)
if prev:
    os.environ['RESUME_ADAPTER'] = os.path.dirname(prev[0])
    print('RESUMING from', os.environ['RESUME_ADAPTER'])
else:
    print('fresh adapter (no previous one attached)')

# --no-smoke: don't spend slow T4 steps re-proving the config; grad is already verified
!python scripts/03_train_lora.py \
    --data-path data/train_sft.jsonl \
    --output-dir /kaggle/working/lora_adapter \
    --no-smoke


## 6. Package the submission — and how to RESUME next session

`submission.zip` (rank 16 ≤ 32 ✓) is built from the latest adapter; download it from
the **Output** tab and submit.

**To train more for free next session:** after this run, click *Save Version →
Save & Run All*; the `/kaggle/working/lora_adapter` folder becomes part of the
notebook output. Create a **Dataset** from that output (or *+ Add Data → your
notebook output*), attach it to a new session, and re-run — the train cell auto-detects
it via `RESUME_ADAPTER` and continues from where you stopped.


In [ ]:
!python scripts/05_package_submission.py --adapter-dir /kaggle/working/lora_adapter --output /kaggle/working/submission.zip